# ζ(2): exact rates and the final bound $\mu(\zeta(2))<4.62675$

This independent notebook supports **Lemma 6.19 (`lem:z2-selected-error`)**, **Lemma 6.21 (`lem:z2-growth`)**, **Theorem 6.2 (`thm:cmf2`)**, and the $\zeta(2)$ case of **Theorem 6.16 (`thm:z23-measures`)**.

It recomputes the spectrum, logarithm bounds, carry integral, and final rational inequalities from the supplied Python certificates. The Barnes-integral bounds, recurrence asymptotics, uniform arithmetic normalization, and measure criterion are mathematical inputs from the paper. Exact numerical certification of their constants is the task of this notebook. No Lean code is generated or run.

In [1]:
from pathlib import Path
import sys
# Works from this notebook folder or from the unzipped bundle root.
HERE = next((p.resolve() for p in [Path.cwd(), Path.cwd()/"zeta2"]
             if (p/"companion.py").is_file()), None)
if HERE is None:
    raise FileNotFoundError("Run this notebook from the zeta2 folder or bundle root.")
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE/"certificates"))
from fractions import Fraction as Q
from IPython.display import display, Markdown
import companion as z2
print("Local Python support loaded. No other notebook needs to be run.")

Local Python support loaded. No other notebook needs to be run.


## 1. A forward-matrix computation of the same spectral gap

The paper presents the error rate through the normalized dual matrix $Q_\infty$:
\[
g=\log\left|\frac{\lambda_1}{\lambda_3}\right|,
\quad \lambda_{1,2}\approx-1.287\pm2.207i,\quad
\lambda_3\approx-1.602\cdot10^{-27}.
\]
The supplied self-contained certificate computes the **forward** matrix at infinity, denoted $A_\infty$ here to distinguish it from $Q_\infty$. The dual pairing in Lemma 6.17 identifies the reciprocal spectra up to a common scalar and conjugacy; consequently their dominant-to-small modulus ratios give the same $g$. The included `dual_pairing.py` reconstructs and verifies that full rational identity from the paper's parameter data; this notebook reruns it independently.

In the forward basis, the limiting $\theta$ matrix is
\[
\Theta_\infty=\begin{pmatrix}0&0&10890/13\\1&0&2305/13\\0&1&42/13\end{pmatrix},
\]
and
\[
A_\infty=
\prod_{u=10,9,11}(I+\Theta_\infty/u)^u\,
(I-\Theta_\infty/11)^{-11}
\prod_{v=2,1,3}(I+\Theta_\infty/v)^{-v}.
\]
All factors are rational functions of one matrix, so their limiting products commute. The program multiplies them exactly and verifies the characteristic polynomial and a nonzero observability determinant for the first-coordinate readout.

In [2]:
from dual_pairing import certify_pairing
pairing=certify_pairing()
assert pairing["identity_verified"]
analytic,seconds=z2.run_certificate("analytic/certify_analytic_rates.py")
assert analytic["cubic_discriminant_negative"]
assert Q(analytic["observability_determinant"]) != 0
print("Exact spectrum/readout certificate regenerated in",seconds,"seconds.")
print("Dominant modulus bracket:",analytic["dominant_modulus_interval"])
print("Observability determinant is nonzero.")

Exact spectrum/readout certificate regenerated in 0.03 seconds.
Dominant modulus bracket: [17400086428, 17400086429]
Observability determinant is nonzero.


## 2. Isolate the dominant root using rational arithmetic

Write $\chi_{A_\infty}(\lambda)=\lambda^3+c_2\lambda^2+c_1\lambda+c_0$. Its negative discriminant gives one real root and one nonreal conjugate pair. Let $\Lambda>0$ be the modulus of the real root, so it is a root of $x^3-c_2x^2+c_1x-c_0$. Sign changes isolate
\[
17400086428<\Lambda<17400086429.
\]
Since the product of the three eigenvalues is $-c_0$, the squared modulus of the nonreal pair is $c_0/\Lambda$. Therefore
\[
g=\log\frac{\Lambda}{\sqrt{c_0/\Lambda}}
=\frac32\log\Lambda-\frac12\log c_0.
\]
The following rechecks the algebra of the saved certificate using exact fractions.

In [3]:
c0,c1,c2,leading=map(Q,analytic["characteristic_ascending"])
L,U=analytic["dominant_modulus_interval"]
root_polynomial=lambda x: x**3-c2*x*x+c1*x-c0
assert leading==1
assert root_polynomial(L)<0<root_polynomial(U)
assert 0<c0/Q(L)<1
print("root polynomial has the required exact sign change.")
print("c0 =",c0)

root polynomial has the required exact sign change.
c0 = 1855425871872/895833302668638834554683750087890625


## 3. Directed logarithm bounds — Lemma 6.19

The logarithm certificate reduces each positive rational to $2^k u$ with $1\le u\le2$ and evaluates
\[
\log u=2\sum_{j=0}^{K-1}\frac{y^{2j+1}}{2j+1}+R_K,
\quad y=\frac{u-1}{u+1},\quad
0\le R_K\le\frac{2y^{2K+1}}{(2K+1)(1-y^2)}.
\]
With $K=32$, every endpoint remains rational. Negative coefficients reverse the interval endpoints. **No floating-point comparison proves a threshold.**

The certificate's field `spectral_gap_lower_enclosure` encloses the lower estimate obtained by replacing $\Lambda$ with $L$; it is not, by itself, a two-sided enclosure of the actual $g$. Below we separately use $U$ to certify an upper bound for $g$.

In [4]:
sys.path.insert(0,str(HERE/"certificates"/"analytic"))
from certify_analytic_rates import linear_logs
lower_gap=linear_logs([(Q(3,2),Q(L)),(Q(-1,2),c0)])[0]
upper_gap=linear_logs([(Q(3,2),Q(U)),(Q(-1,2),c0)])[1]
assert lower_gap>Q("62.636575")
assert upper_gap<Q("62.636576")
print(float(lower_gap),"< g <",float(upper_gap))
print("Hence the paper's g > 62.636575 is certified.")

62.636575041510795 < g < 62.636575041597
Hence the paper's g > 62.636575 is certified.


The common factorial normalization in the dual system is $(n!)^{13}$. The scalar pairing factor and this factorial factor both cancel when dividing the error numerator by $q_n$. The recessiveness and dominant-component cancellation proved in Appendix `app:zeta2-proof` / `app:z2-error` then give
\[
\left|\zeta(2)-\frac{p_n}{q_n}\right|\le e^{-(g-o(1))n}.
\]
The supplied analytic certificate also verifies a coarse contour exponent below $-18$. This supports the written analytic estimates; it is not a numerical integration substituted for the contour argument.

In [5]:
def enclosure(name):
    row=analytic[name]
    return Q(row["lower"]),Q(row["upper"])
coefficient_bounds=enclosure("scaled_coefficient_exponent_upper_enclosure")
contour_bounds=enclosure("coarse_contour_exponent")
assert coefficient_bounds[1] < Q("38.168733")
assert contour_bounds[1] < -18
print("Growth rate of C_n q_n <",float(coefficient_bounds[1]))
print("Coarse contour exponent <",float(contour_bounds[1]))

Growth rate of C_n q_n < 38.16873290964669
Coarse contour exponent < -18.8496451210932


## 4. Arithmetic growth — Lemma 6.21

To keep this notebook independent, recompute the carry-profile integral rather than relying on notebook 02. The paper's six shells have total rate $233/120$, and the 94-cell, 2000-period certificate gives $\gamma>12.1284279747910$ with a directed rational tail bound. Thus
\[
h<38.168733+25-\left(12.1284279747910+\frac{233}{120}\right)
<49.098639.
\]
The exact carry certificate proves the slightly more informative inequality $25-\gamma-233/120<10.929906$. We use the paper's displayed bounds in the final calculation.

**Rounding correction to the manuscript's displayed chain:** using only $\gamma>12.128427$ gives $49.098639333\ldots$, which is slightly above $49.098639$. The regenerated exact certificate is stronger and justifies the stated $h<49.098639$ without changing the final theorem.

In [6]:
rate,seconds=z2.run_certificate("rate_certificate.py")
gamma_lower=Q(rate["gamma_lower"])
shell_rate=sum((b-a for a,b in z2.SHELLS),Q(0))
assert shell_rate==Q(233,120)
assert gamma_lower>Q("12.128427")
arithmetic_upper=25-gamma_lower-shell_rate
assert arithmetic_upper<Q("10.929906")
h_upper=Q("38.168733")+Q("10.929906")
assert h_upper==Q("49.098639")
assert Q("38.168733")+arithmetic_upper<h_upper
# The rounded display in the manuscript loses 1/3,000,000 here.
rounded_chain=Q("38.168733")+25-Q("12.128427")-shell_rate
assert rounded_chain-h_upper == Q(1,3_000_000)
print("h <",float(h_upper))

h < 49.098639


## 5. Final comparison — Theorems 6.2 and 6.16

After Lemma 6.20 the pairs $(\widetilde p_n,\widetilde q_n)$ are integers. Their ratio is $p_n/q_n$, and $|\widetilde q_n|\le e^{(h+o(1))n}$. The nonvanishing input in the paper is the known irrationality of $\zeta(2)$: a rational approximant cannot equal it.

For $g>h>0$ the paper's criterion gives
\[
\delta=\frac gh-1,\qquad
\mu(\zeta(2))\le\frac{g}{g-h}.
\]
The function $g/(g-h)$ decreases as $g$ increases and increases as $h$ increases. Therefore substituting a lower bound for $g$ and an upper bound for $h$ is the safe direction. Keep the exact printed decimal thresholds as fractions.

In [7]:
g_lower=Q("62.636575")
h_upper=Q("49.098639")
gap_lower=g_lower-h_upper
delta_lower=g_lower/h_upper-1
mu_upper=g_lower/gap_lower
assert gap_lower==Q("13.537936")
assert delta_lower>Q("0.2757")
assert mu_upper<Q("4.62675")
assert 1+1/delta_lower==mu_upper
print("g − h >",float(gap_lower))
print("delta >",float(delta_lower))
print("mu(zeta(2)) <",float(mu_upper),"< 4.62675")

g − h > 13.537936
delta > 0.27572935371996765
mu(zeta(2)) < 4.626744800684536 < 4.62675


The conclusion matches the paper: $\delta>0.2757$ and $\mu(\zeta(2))<4.62675$.

**Certificate boundary.** This notebook recomputes exact finite algebra and interval/rational inequalities. The all-depth integer normalization, asymptotic recurrence theory, Barnes-integral representation, prime number theorem, and the irrationality-measure criterion remain the paper's stated mathematical inputs. The supplied Lean material is treated as reference only.